# LSTM embeddings and the contribution of possession history

This experiment compares a forward LSTM that carries history within each
possession with an LSTM reset at every event. Separate models predict a subsequent shot
and a same-team goal, using the existing eligible states before the first shot.

All four models tune on **2025**, refit on **2023–2025** and evaluate the existing
**2026 benchmark**. This notebook reads completed artifacts; it does not train,
tune or rescore test inputs. Saved outputs are included. Run cells in order using
the project kernel. See [the experiment specification](../docs/lstm-baseline.md).

## 1. Experiment verification

The loader resolves repository paths and verifies every saved artifact
against the completion manifest. Missing completion metadata means the run has not finished successfully.
The protocol records the candidate set and fitting rules chosen before test loading.

In [1]:
import hashlib
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

project_root=next((p for p in [Path.cwd(),*Path.cwd().parents]
                   if (p/'data/sources.json').is_file() and (p/'pyproject.toml').is_file()),None)
if project_root is None:
    raise RuntimeError("Launch from the repository root or notebooks folder.")
experiment=project_root/'artifacts/lstm-2026'
if not (experiment/'summary.json').is_file():
    raise RuntimeError("Complete python -m afl_opposition.lstm_experiment first; use --resume if interrupted.")
summary=json.loads((experiment/'summary.json').read_text())
protocol=json.loads((experiment/'protocol.json').read_text())
for relative,expected in summary['artifact_sha256'].items():
    with (experiment/relative).open('rb') as stream:
        assert hashlib.file_digest(stream,'sha256').hexdigest()==expected,relative
targets=protocol['targets']; variants=protocol['variants']
refits={(t,v):json.loads((experiment/t/v/'refit.json').read_text()) for t in targets for v in variants}
print('Experiment:',experiment.relative_to(project_root).as_posix())
print('Verified artifacts:',len(summary['artifact_sha256']))
print('PyTorch:',protocol['torch_version'],'Device:',summary['device'])

Experiment: artifacts/lstm-2026
Verified artifacts: 224
PyTorch: 2.11.0+cu128 Device: cuda


## 2. Selected architectures

Each event combines an action embedding with four scaled numeric features. A
forward LSTM produces a hidden vector at that event; dropout and a linear head
produce a logit. Sigmoid gives the reported probability. Output dropout applies
even with one layer; deeper networks also use it between recurrent layers.

`history` carries memory within a possession. `no_history` runs every event as a
length-one sequence with zero initial state. Both reset at possession boundaries.
Validation selects layer count, hidden size, embedding dimension and dropout. The table shows the validation winners.

In [2]:
selected=[]
for (target,variant),record in refits.items():
    selected.append(dict(target=target,variant=variant,**record['config'],epochs=record['epochs'],
        parameters=record['selected']['parameters'],validation_bce=record['selected']['validation_bce']))
display(pd.DataFrame(selected).style.format({'validation_bce':'{:.6f}','lambda_l2':'{:.2g}'}))

,target,variant,num_layers,hidden_size,dropout,embedding_dim,learning_rate,lambda_l2,epochs,parameters,validation_bce
0,shot,no_history,2,32,0.000000,16,0.003000,0,27,15617,0.390954
1,shot,history,2,32,0.000000,16,0.003000,0,27,15617,0.384916
2,goal,no_history,1,32,0.000000,16,0.003000,0,43,7169,0.278884
3,goal,history,2,32,0.000000,16,0.003000,0,27,15617,0.276273


## 3. Example input sequence

The example contains a four-action possession from the development labels. Each row
adds observed information. At event 2, the history model can use events 0, 1 and 2;
the control receives only event 2. Both predict the same eventual possession
outcome. Shot flags, future events and outcome annotations are absent from inputs.

The category IDs below use the goal/history model's **training-only** encoder.
IDs 0 and 1 are reserved for padding and unknown actions. Numeric features remain
continuous; they are standardised inside the model input pipeline.

In [3]:
development=pd.read_parquet(project_root/'artifacts/goal-labels-2023-2025/examples.parquet')
sizes=development.groupby('possession_id').size()
chosen=sizes.loc[sizes.eq(4)].index[0]
example=development.loc[development.possession_id.eq(chosen)].sort_values('event_index').copy()
encoder=json.loads((experiment/'goal/history/selection/model.json').read_text())['encoder']
vocabulary={name:i+2 for i,name in enumerate(encoder['categories'])}
example['action_id']=example.action_type.map(vocabulary).fillna(1).astype(int)
display(example[['event_index','action_type','action_id','x_norm','y_norm','gap_seconds','elapsed_seconds','target','sample_weight']])
display(sizes.describe(percentiles=[.5,.9,.99]).rename('eligible sequence length').to_frame())
del development

,event_index,action_type,action_id,x_norm,y_norm,gap_seconds,elapsed_seconds,target,sample_weight
30,0,gather,4,0.7375,-0.070922,0,0,0,0.25
31,1,handball,6,0.7,-0.042553,1,1,0,0.25
32,2,receive,13,0.5625,0.056738,1,2,0,0.25
33,3,kick,8,0.5875,0.113475,1,3,0,0.25


,eligible sequence length
count,153822.000000
mean,5.491900
std,4.208851
min,1.000000
50%,4.000000
90%,11.000000
99%,20.000000
max,47.000000


## 4. Training objective and search budget

The loss is the mean of the within-possession mean BCE, plus explicit L2 on
non-bias weight matrices:

$$J=\frac1N\sum_i\frac1{n_i}\sum_t\mathrm{BCE}(y_i,p_{i,t})
+\lambda\|\theta_{\rm reg}\|_2^2.$$

Every possession has total weight one, regardless of sequence length. Padding
receives zero weight. Adam uses zero weight decay: the L2 term is part of the
objective. Validation selection uses BCE alone, with dropout disabled.

All four searches receive the same 24 candidate configurations. Candidates
train for up to 6 epochs; the best 6 continue to 18, and the best 2 to 48. Early stopping uses
six epochs without improvement. Continuing a stage restores optimiser and random
state as well as weights. This bounded search can prune a slow-learning candidate.

In [4]:
display(pd.DataFrame(protocol['candidates']).rename_axis('trial'))
display(pd.DataFrame(protocol['stages'],columns=['cumulative_epoch_budget','candidates_retained']))
print('Batch size:',protocol['batch_size'],'possessions')
print('Training seed:',protocol['seed'],'Early-stopping patience:',protocol['patience'])
print('Optimiser:',protocol['optimiser'])

,num_layers,hidden_size,dropout,embedding_dim,learning_rate,lambda_l2
trial,,,,,,
0,1,64,0.10,8,0.0010,0.000010
1,3,64,0.10,16,0.0030,0.000001
2,3,32,0.40,4,0.0010,0.000000
3,1,128,0.00,16,0.0003,0.000010
4,3,32,0.10,4,0.0030,0.000010
5,2,32,0.25,8,0.0003,0.000001
6,1,64,0.10,16,0.0003,0.000001
7,2,32,0.25,8,0.0010,0.000010
8,1,128,0.40,16,0.0030,0.000100


,cumulative_epoch_budget,candidates_retained
0,6,6
1,18,2
2,48,1


Batch size: 1024 possessions
Training seed: 2026 Early-stopping patience: 6
Optimiser: Adam with zero weight_decay; explicit L2


## 5. Candidate validation results

The complete search tables retain every candidate, including pruned ones. The
table below shows the five best observed candidates for each target and variant.
`stage` and `attempted_epochs` expose their different training budgets. Selection
uses the best validation checkpoint, with exact ties favouring fewer parameters
and then the earlier trial index. The search is independent for each model.

In [5]:
searches={key:pd.read_parquet(experiment/key[0]/key[1]/'search.parquet') for key in refits}
for (target,variant),search in searches.items():
    assert search.selected.sum()==1
    assert search.loc[search.selected,'validation_bce'].iloc[0]==search.validation_bce.min()
    print(target,variant)
    display(search.sort_values('validation_bce').head(5)[['trial','num_layers','hidden_size','embedding_dim','dropout','learning_rate','lambda_l2','stage','attempted_epochs','best_epoch','validation_bce','selected']]
            .style.format({'validation_bce':'{:.6f}','lambda_l2':'{:.2g}'}))

shot no_history


,trial,num_layers,hidden_size,embedding_dim,dropout,learning_rate,lambda_l2,stage,attempted_epochs,best_epoch,validation_bce,selected
21,21,2,32,16,0.000000,0.003000,0,3,33,27,0.390954,True
20,20,2,64,4,0.250000,0.003000,1e-06,3,33,27,0.391440,False
4,4,3,32,4,0.100000,0.003000,1e-05,2,18,17,0.392873,False
1,1,3,64,16,0.100000,0.003000,1e-06,2,13,7,0.393640,False
12,12,1,32,16,0.000000,0.003000,0,2,18,18,0.393659,False


shot history


,trial,num_layers,hidden_size,embedding_dim,dropout,learning_rate,lambda_l2,stage,attempted_epochs,best_epoch,validation_bce,selected
21,21,2,32,16,0.000000,0.003000,0,3,33,27,0.384916,True
12,12,1,32,16,0.000000,0.003000,0,3,33,27,0.386401,False
11,11,1,128,8,0.250000,0.003000,1e-05,2,18,18,0.389829,False
20,20,2,64,4,0.250000,0.003000,1e-06,2,13,7,0.391036,False
4,4,3,32,4,0.100000,0.003000,1e-05,2,13,7,0.392158,False


goal no_history


,trial,num_layers,hidden_size,embedding_dim,dropout,learning_rate,lambda_l2,stage,attempted_epochs,best_epoch,validation_bce,selected
12,12,1,32,16,0.000000,0.003000,0,3,48,43,0.278884,True
21,21,2,32,16,0.000000,0.003000,0,3,33,27,0.279059,False
1,1,3,64,16,0.100000,0.003000,1e-06,2,18,15,0.279764,False
4,4,3,32,4,0.100000,0.003000,1e-05,2,18,17,0.280318,False
11,11,1,128,8,0.250000,0.003000,1e-05,2,18,18,0.280816,False


goal history


,trial,num_layers,hidden_size,embedding_dim,dropout,learning_rate,lambda_l2,stage,attempted_epochs,best_epoch,validation_bce,selected
21,21,2,32,16,0.000000,0.003000,0,3,33,27,0.276273,True
12,12,1,32,16,0.000000,0.003000,0,3,33,27,0.276388,False
1,1,3,64,16,0.100000,0.003000,1e-06,2,13,7,0.278231,False
20,20,2,64,4,0.250000,0.003000,1e-06,2,13,7,0.278850,False
4,4,3,32,4,0.100000,0.003000,1e-05,2,13,7,0.279590,False


## 6. Selected-model learning curves

The table includes the first, every fourth, best and final attempted epochs. Training BCE is
the online average while minibatches update the model and dropout is active.
Validation BCE is evaluated after the epoch with fixed weights and dropout off;
the two numbers are therefore measured under different conditions.

The best epoch determines refit duration. `best_at_budget` indicates that the
selected checkpoint is at the declared budget rather than an interior minimum.

In [6]:
for (target,variant),record in refits.items():
    chosen=record['selected']
    curve=pd.read_parquet(experiment/target/variant/'learning_curves.parquet')
    curve=curve.loc[curve.trial.eq(chosen['trial'])]
    mask=curve.epoch.isin([1,chosen['best_epoch'],chosen['attempted_epochs']]) | curve.epoch.mod(4).eq(0)
    shown=curve.loc[mask,['epoch','training_bce','validation_bce','seconds']].copy()
    shown['selected_epoch']=shown.epoch.eq(chosen['best_epoch'])
    print(target,variant,'— best at budget:',chosen['best_at_budget'])
    display(shown.style.format({'training_bce':'{:.6f}','validation_bce':'{:.6f}','seconds':'{:.2f}'}))

shot no_history — best at budget: False


,epoch,training_bce,validation_bce,seconds,selected_epoch
196,1,0.455764,0.418719,0.53,False
199,4,0.395706,0.400341,0.53,False
203,8,0.387741,0.395269,0.51,False
207,12,0.386022,0.395351,0.51,False
211,16,0.384582,0.394336,0.52,False
215,20,0.383360,0.391592,0.52,False
219,24,0.384347,0.391394,0.53,False
222,27,0.384397,0.390954,0.50,True
223,28,0.382703,0.392952,0.52,False
227,32,0.382249,0.394569,0.50,False


shot history — best at budget: False


,epoch,training_bce,validation_bce,seconds,selected_epoch
186,1,0.464420,0.437341,0.96,False
189,4,0.394089,0.403557,0.92,False
193,8,0.383813,0.390538,0.97,False
197,12,0.382313,0.393868,0.97,False
201,16,0.379545,0.389407,0.95,False
205,20,0.378123,0.385969,0.97,False
209,24,0.379533,0.385624,0.97,False
212,27,0.378474,0.384916,0.95,True
213,28,0.377245,0.386529,0.95,False
217,32,0.376337,0.390439,0.99,False


goal no_history — best at budget: False


,epoch,training_bce,validation_bce,seconds,selected_epoch
108,1,0.299995,0.289591,0.46,False
111,4,0.277656,0.282944,0.45,False
115,8,0.275409,0.281319,0.45,False
119,12,0.274645,0.280789,0.46,False
123,16,0.274012,0.280101,0.43,False
127,20,0.273131,0.279391,0.44,False
131,24,0.273392,0.279289,0.43,False
135,28,0.272765,0.279601,0.43,False
139,32,0.272486,0.280118,0.43,False
143,36,0.272859,0.279276,0.43,False


goal history — best at budget: False


,epoch,training_bce,validation_bce,seconds,selected_epoch
181,1,0.304986,0.296213,0.99,False
184,4,0.276801,0.284378,0.98,False
188,8,0.273009,0.278483,0.96,False
192,12,0.272339,0.281375,0.95,False
196,16,0.271098,0.278120,0.96,False
200,20,0.270479,0.276567,0.96,False
204,24,0.271373,0.276710,0.94,False
207,27,0.270678,0.276273,0.94,True
208,28,0.270000,0.277024,0.95,False
212,32,0.269498,0.278859,0.97,False


## 7. Refitting and test coverage

Each final model and encoder are fitted from scratch on 2023–2025 for the selected
number of epochs. There is no validation-based stopping during refitting. The
selected mean-loss L2 coefficient remains fixed. All four models are saved before
the first 2026 load, and their parameters remain unchanged during scoring.

Test event IDs, labels, weights and constant probabilities are checked against the
previous baselines. Counts below refer to correlated event states within possessions.

In [7]:
assert summary['all_four_models_frozen_before_test_loading']
assert summary['models_unchanged_during_evaluation']
predictions={key:pd.read_parquet(experiment/key[0]/key[1]/'test/predictions.parquet') for key in refits}
coverage=[]
for (target,variant),table in predictions.items():
    assert table.groupby('possession_id').sample_weight.sum().sub(1).abs().max()<1e-12
    record=refits[(target,variant)]
    assert record['epochs']==record['selected']['best_epoch']
    assert record['config']['lambda_l2']==record['selected']['lambda_l2']
    coverage.append(dict(target=target,variant=variant,refit_possessions=record['refit_possessions'],
        test_matches=table.match_id.nunique(),test_possessions=table.possession_id.nunique(),test_prefixes=len(table),
        test_positive_rate=(table.target*table.sample_weight).sum()/table.sample_weight.sum()))
display(pd.DataFrame(coverage))

,target,variant,refit_possessions,test_matches,test_possessions,test_prefixes,test_positive_rate
0,shot,no_history,153822,218,51234,290909,0.216341
1,shot,history,153822,218,51234,290909,0.216341
2,goal,no_history,153822,218,51234,290909,0.105809
3,goal,history,153822,218,51234,290909,0.105809


## 8. Five-model comparison on the 2026 benchmark

Lower BCE and Brier are better; higher AP is better. All metrics in this table use
possession weights. Precision, recall and F1 use the fixed threshold 0.5. Comparisons are
within each target because shot and goal prevalence differ. The constant
predictor's AP equals test positive prevalence, not 0.5.

In [8]:
comparisons={target:pd.read_parquet(experiment/target/'comparison/metrics.parquet') for target in targets}
metrics=pd.concat([table.assign(target=target) for target,table in comparisons.items()],ignore_index=True)
display(metrics[['target','model','log_loss','brier_score','average_precision','accuracy','precision','recall','f1','balanced_accuracy']]
        .style.format({c:'{:.6f}' for c in ['log_loss','brier_score','average_precision','accuracy','precision','recall','f1','balanced_accuracy']},na_rep='undefined'))

,target,model,log_loss,brier_score,average_precision,accuracy,precision,recall,f1,balanced_accuracy
0,shot,constant,0.523087,0.169815,0.216341,0.783659,undefined,0.000000,0.000000,0.500000
1,shot,logistic,0.423148,0.134152,0.555929,0.815618,0.672837,0.287535,0.402895,0.624469
2,shot,xgboost,0.398320,0.125400,0.612536,0.829024,0.756987,0.308838,0.438695,0.640734
3,shot,lstm_no_history,0.401857,0.126606,0.606363,0.827301,0.724644,0.325362,0.449087,0.645616
4,shot,lstm_history,0.394988,0.124298,0.618982,0.829832,0.716298,0.353392,0.473285,0.657376
5,goal,constant,0.337951,0.094666,0.105809,0.894191,undefined,0.000000,0.000000,0.500000
6,goal,logistic,0.297569,0.085786,0.284618,0.894596,0.589756,0.012575,0.024625,0.505770
7,goal,xgboost,0.287580,0.082488,0.342662,0.899497,0.657114,0.104869,0.180872,0.549197
8,goal,lstm_no_history,0.288036,0.082635,0.340169,0.899156,0.657531,0.097918,0.170452,0.545941
9,goal,lstm_history,0.285696,0.082102,0.347233,0.899349,0.634094,0.115250,0.195049,0.553690


## 9. Raw and weighted confusion matrices

Rows are actual classes and columns predicted classes: `[[TN, FP], [FN, TP]]`.
Raw entries count event states. Weighted entries sum prefix weights; their total
equals the number of eligible possessions, but there is still a prediction at
every event. Undefined precision means the model made no positive predictions.

In [9]:
for target in targets:
    table=pd.read_parquet(experiment/target/'comparison/confusion.parquet')
    for variant in variants:
        for weighting in ['unweighted','possession_weighted']:
            row=table.loc[table.model.eq('lstm_'+variant)&table.weighting.eq(weighting)].iloc[0]
            matrix=pd.DataFrame([[row.tn,row.fp],[row.fn,row.tp]],
                index=[f'Actual no {target}',f'Actual {target}'],columns=[f'Predicted no {target}',f'Predicted {target}'])
            print(target,variant,weighting,'— threshold 0.5')
            display(matrix.style.format('{:.0f}' if weighting=='unweighted' else '{:.2f}'))

shot no_history unweighted — threshold 0.5


,Predicted no shot,Predicted shot
Actual no shot,201058,7737
Actual shot,64609,17505


shot no_history possession_weighted — threshold 0.5


,Predicted no shot,Predicted shot
Actual no shot,38779.64,1370.36
Actual shot,7477.68,3606.32


shot history unweighted — threshold 0.5


...

,Predicted no goal,Predicted goal
Actual no goal,45536.53,276.47
Actual goal,4890.19,530.81


goal history unweighted — threshold 0.5


,Predicted no goal,Predicted goal
Actual no goal,247653,1872
Actual goal,38225,3159


goal history possession_weighted — threshold 0.5


,Predicted no goal,Predicted goal
Actual no goal,45452.47,360.53
Actual goal,4796.23,624.77


## 10. Contribution of event history

The test comparison below uses each variant's independently tuned winner, so the
architectures can differ. It measures the best observed versions in this search.
The matched-candidate validation comparison also holds architecture, optimiser
settings and the initial six-epoch budget fixed. Negative BCE differences favour
the model with history. Neither comparison proves a causal football effect.

In [10]:
changes=[]
for target,table in comparisons.items():
    indexed=table.set_index('model')
    changes.append(dict(target=target,
        history_minus_control_BCE=indexed.loc['lstm_history','log_loss']-indexed.loc['lstm_no_history','log_loss'],
        history_minus_control_AP=indexed.loc['lstm_history','average_precision']-indexed.loc['lstm_no_history','average_precision'],
        history_minus_XGBoost_BCE=indexed.loc['lstm_history','log_loss']-indexed.loc['xgboost','log_loss']))
display(pd.DataFrame(changes).style.format({c:'{:+.6f}' for c in ['history_minus_control_BCE','history_minus_control_AP','history_minus_XGBoost_BCE']}))
for target in targets:
    matched={}
    for variant in variants:
        curves=pd.read_parquet(experiment/target/variant/'learning_curves.parquet')
        matched[variant]=curves.loc[curves.epoch.le(6)].groupby('trial').validation_bce.min()
    table=pd.DataFrame(matched)
    table['history_minus_control']=table.history-table.no_history
    print(target,'— matched candidates, best BCE within first six epochs')
    display(table.style.format('{:.6f}'))

,target,history_minus_control_BCE,history_minus_control_AP,history_minus_XGBoost_BCE
0,shot,-0.006869,+0.012620,-0.003332
1,goal,-0.002340,+0.007064,-0.001884


shot — matched candidates, best BCE within first six epochs


,no_history,history,history_minus_control
trial,,,
0,0.405922,0.406698,0.000776
1,0.399421,0.399650,0.000229
2,0.411525,0.419357,0.007832
3,0.414350,0.430159,0.015808
4,0.399950,0.397001,-0.002949
5,0.419729,0.449721,0.029991
6,0.427540,0.441153,0.013613
7,0.406929,0.412573,0.005644
8,0.401271,0.396611,-0.004661


goal — matched candidates, best BCE within first six epochs


,no_history,history,history_minus_control
trial,,,
0,0.285515,0.284231,-0.001284
1,0.282221,0.281402,-0.000819
2,0.287849,0.290457,0.002609
3,0.288220,0.292745,0.004524
4,0.283143,0.281536,-0.001606
5,0.290186,0.301804,0.011618
6,0.291579,0.297094,0.005516
7,0.286065,0.286706,0.000641
8,0.284186,0.282318,-0.001868


## 11. Calibration and interpretation limits

Calibration bands compare weighted mean predictions with observed outcomes. Agreement is desirable; high bands may contain little weight. These are
diagnostics, not a calibration fit or a threshold search.

All neural candidates used one fixed training seed. The search is bounded and
the 2026 benchmark was already observed for earlier models. These results do not
establish statistical significance, embedding superiority over one-hot encoding,
or live match usefulness. Further seed/temporal validation would strengthen a
deployment claim. The saved artifacts preserve the full experiment for review.

In [11]:
for target in targets:
    table=pd.read_parquet(experiment/target/'comparison/calibration.parquet')
    table=table.loc[table.model.str.startswith('lstm_')].copy()
    observed=f'observed_{target}_rate'
    table['prediction_minus_observed']=table.mean_prediction-table[observed]
    print(target)
    display(table[['model','bin_lower','bin_upper','examples','weight_sum','mean_prediction',observed,'prediction_minus_observed']]
        .style.format({'weight_sum':'{:.1f}','mean_prediction':'{:.4f}',observed:'{:.4f}','prediction_minus_observed':'{:+.4f}'}))

shot


,model,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_shot_rate,prediction_minus_observed
21,lstm_no_history,0.000000,0.100000,99527,20179.5,0.0492,0.0575,-0.0084
22,lstm_no_history,0.100000,0.200000,72430,11944.3,0.1506,0.1603,-0.0098
23,lstm_no_history,0.200000,0.300000,51537,7989.2,0.2448,0.2601,-0.0153
24,lstm_no_history,0.300000,0.400000,28301,4066.0,0.3446,0.3509,-0.0063
25,lstm_no_history,0.400000,0.500000,13872,2078.3,0.4442,0.4312,+0.0130
26,lstm_no_history,0.500000,0.600000,7024,1133.7,0.5449,0.5158,+0.0290
27,lstm_no_history,0.600000,0.700000,4420,848.7,0.6503,0.5941,+0.0562
28,lstm_no_history,0.700000,0.800000,4470,912.8,0.7498,0.6604,+0.0894
29,lstm_no_history,0.800000,0.900000,2984,622.3,0.8413,0.7804,+0.0609
30,lstm_no_history,0.900000,1.000000,6344,1459.1,0.9730,0.9792,-0.0062


goal


,model,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_goal_rate,prediction_minus_observed
20,lstm_no_history,0.000000,0.100000,167632,31761.8,0.0481,0.0472,+0.0009
21,lstm_no_history,0.100000,0.200000,84480,13408.5,0.1392,0.1420,-0.0028
22,lstm_no_history,0.200000,0.300000,25418,3537.9,0.2407,0.2360,+0.0047
23,lstm_no_history,0.300000,0.400000,6670,1178.7,0.3434,0.3483,-0.0049
24,lstm_no_history,0.400000,0.500000,2958,539.9,0.4425,0.4477,-0.0052
25,lstm_no_history,0.500000,0.600000,1478,319.1,0.5468,0.5309,+0.0158
26,lstm_no_history,0.600000,0.700000,1127,258.4,0.6480,0.6612,-0.0132
27,lstm_no_history,0.700000,0.800000,972,202.6,0.7462,0.8159,-0.0697
28,lstm_no_history,0.800000,0.900000,174,27.2,0.8148,0.9281,-0.1133
29,lstm_history,0.000000,0.100000,170345,32722.3,0.0440,0.0474,-0.0034
